# Hailmary cluster artifact audit

This notebook is a read-only, reproducible walkthrough of a cluster-library artifact created by `hailmary-build-clusters`. Set `HAILMARY_CLUSTER_ARTIFACT` to select an artifact; no clustering is refit in the notebook.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np

from hailmary.clustering import ClusterLibrary

artifact_path = Path(os.environ.get("HAILMARY_CLUSTER_ARTIFACT", "artifacts/hailmary/clusters.json"))
artifact_path

## 1. Load and verify the content-addressed artifact

`ClusterLibrary.read` recomputes and verifies the canonical content hash.

In [ ]:
library = ClusterLibrary.read(artifact_path)
{
    "dataset_id": library.dataset_id,
    "partition": f"{library.airport}/{library.runway}",
    "station_count": library.resample_station_count,
    "cluster_count": len(library.medoids),
    "assignment_count": len(library.assignments),
    "algorithm": library.clustering.algorithm,
    "used_fallback": library.clustering.used_fallback,
    "content_hash": library.artifact_content_hash,
}

## 2. Inspect the observed medoid geometries

Coordinates are runway-local metres and tracks run from terminal entry toward the threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for medoid in library.medoids:
    points = medoid.points_m
    ax.plot(points[:, 0] / 1852.0, points[:, 1] / 1852.0, label=f"C{medoid.cluster_id}: {medoid.medoid_flight_id}")
ax.scatter([0.0], [0.0], marker="*", s=120, color="black", label="runway threshold")
ax.set(xlabel="east (NM)", ylabel="north (NM)", title="Cluster medoid paths")
ax.axis("equal")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
plt.show()

## 3. Audit noise reassignment and OOD provenance

In [ ]:
cluster_ids = sorted({item.cluster_id for item in library.assignments})
counts = [sum(item.cluster_id == cluster_id for item in library.assignments) for cluster_id in cluster_ids]
noise_counts = [sum(item.cluster_id == cluster_id and item.was_hdbscan_noise for item in library.assignments) for cluster_id in cluster_ids]
ood_counts = [sum(item.cluster_id == cluster_id and item.out_of_distribution for item in library.assignments) for cluster_id in cluster_ids]

x = np.arange(len(cluster_ids))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x, counts, label="all final assignments")
ax.bar(x, noise_counts, label="HDBSCAN noise reassigned")
ax.scatter(x, ood_counts, marker="x", color="black", label="OOD count")
ax.set(xticks=x, xticklabels=cluster_ids, xlabel="cluster ID", ylabel="flights", title="Assignment provenance audit")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()